# Comparison of samples

Apply to summary files (output from summary_celltypes_IHOPE.py) in CSV format.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    print_numeric_summary,
    pivot_for_tissue_heatmap,
    plot_celltype_clustermap,
    plot_celltype_stacked_barplot,
    plot_celltype_stripplot,
    normalise_to_parent,
    parent_ratio_labels,
)


Define file names and sample names. Use the output from cell 1 to complete cell 2.

In [ ]:
# Make a "fill the gap" template
summaries_dir = Path("../results/reports/zscore_log2")
files = sorted(summaries_dir.glob("celltype_summary_*.csv"))

print("name_map = {")
for f in files:
    print(f'    "{f.name}": "",')
print("}")

In [ ]:
name_map = {
    "celltype_summary_IHOPE14_MedLN_BottomLeft.csv": "IHOPE14 MedLN Bottom Left",
    "celltype_summary_IHOPE14_MedLN_BottomRight.csv": "IHOPE14 MedLN Bottom Right",
    "celltype_summary_IHOPE14_MedLN_TopRight.csv": "IHOPE14 MedLN Top Right",
    "celltype_summary_IHOPE14_MesLN.csv": "IHOPE14 MesLN",
    "celltype_summary_IHOPE20_MedLN.csv": "IHOPE20 MedLN",
    "celltype_summary_IHOPE20_Spleen.csv": "IHOPE20 Spleen",
    "celltype_summary_IHOPE26_MedLN.csv": "IHOPE26 MedLN",
    "celltype_summary_IHOPE26_Spleen.csv": "IHOPE26 Spleen",
    "celltype_summary_IHOPE27_MedLN.csv": "IHOPE27 MedLN",
    "celltype_summary_IHOPE27_Spleen.csv": "IHOPE27 Spleen",
    "celltype_summary_IHOPE39_MedLN.csv": "IHOPE39 MedLN",
    "celltype_summary_IHOPE39_MesLN_A.csv": "IHOPE39 MesLN A",
    "celltype_summary_IHOPE39_MesLN_B.csv": "IHOPE39 MesLN B",
    "celltype_summary_IHOPE39_Spleen.csv": "IHOPE39 Spleen",
}

In [ ]:
file_map = {
    sample: summaries_dir / fname
    for fname, sample in name_map.items()
}

In [ ]:
# Sanity check
for sample, path in file_map.items():
    print(sample, ": ", path.name)

Generate the combined data frame:

In [ ]:
df = load_celltype_summaries(file_map=file_map)

### Donor-grouping:

In [ ]:
donor_map = {
    "IHOPE14 MedLN Bottom Left": "IHOPE14",
    "IHOPE14 MedLN Bottom Right": "IHOPE14",
    "IHOPE14 MedLN Top Right": "IHOPE14",
    "IHOPE14 MesLN": "IHOPE14",
    "IHOPE20 MedLN": "IHOPE20",
    "IHOPE20 Spleen": "IHOPE20",
    "IHOPE26 MedLN": "IHOPE26",
    "IHOPE26 Spleen": "IHOPE26",
    "IHOPE27 MedLN": "IHOPE27",
    "IHOPE27 Spleen": "IHOPE27",
    "IHOPE39 MedLN": "IHOPE39",
    "IHOPE39 MesLN A": "IHOPE39",
    "IHOPE39 MesLN B": "IHOPE39",
    "IHOPE39 Spleen": "IHOPE39",
}

df["donor"] = df["sample"].map(donor_map)

Donor color code map

In [ ]:
donor_ids = sorted(set(donor_map.values()))
donor_colors = dict(zip(donor_ids, sns.color_palette("colorblind", n_colors=len(donor_ids))))

### Tissue-grouping:

In [ ]:
tissue_map = {
    "IHOPE14 MedLN Bottom Left": "MedLN",
    "IHOPE14 MedLN Bottom Right": "MedLN",
    "IHOPE14 MedLN Top Right": "MedLN",
    "IHOPE14 MesLN": "MesLN",
    "IHOPE20 MedLN": "MedLN",
    "IHOPE20 Spleen": "Spleen",
    "IHOPE26 MedLN": "MedLN",
    "IHOPE26 Spleen": "Spleen",
    "IHOPE27 MedLN": "MedLN",
    "IHOPE27 Spleen": "Spleen",
    "IHOPE39 MedLN": "MedLN",
    "IHOPE39 MesLN A": "MesLN",
    "IHOPE39 MesLN B": "MesLN",
    "IHOPE39 Spleen": "Spleen",
}

df["tissue"] = df["sample"].map(tissue_map)

Using mean  (avoid dominance of large samples):

In [ ]:
df_tissue = (
    df.groupby(["tissue", "level", "cell_type"], as_index=False)
      .agg(pct_total=("pct_total", "mean"))
)

Decide how you want to handle unannotated cells:

In [ ]:
DROP_UNCLASSIFIED = True   # drop 'unclassified' and rescale to 100, applied in the matrix and barplot calls below
HIDE_UNCLASSIFIED = False  # keep percentages as-is, just drop the unclassified row from heatmaps

Order of cell types for heatmaps:

In [ ]:
# Structural / non-immune cell types, dropped when immune_only=True
structural_cell_types = [
    "Blood_Endothelial",
    "Lymphatic_Endothelial",
    "Basement_Membrane",
    "Fibroblast",
    "Stromal",
    "Endothelial",
    "FDC"
]

# Lineage subsets for the per-lineage breakdown barplots
lineage_subsets = {
    "T": [
        "Activated_CD4", "Activated_CD8", "TCM_CD4", "TCM_CD8",
        "TEM_CD4", "TEM_CD8", "TEMRA_CD4", "TEMRA_CD8",
        "TN_CD4", "TN_CD8", "Treg", "TfH_like", "T_terminal",
    ],
    "B": [
        # marker calls, overwritten in place with follicle location
        "B_naive", "B_GC", "B_Plasmablast",
    ],

    "Myeloid": [
        "Monocyte_Macrophage", "cDC1", "cDC2",
    ],
}

# Manual row order for heatmaps, grouping related subtypes together.
celltype_order = [
    # type level
    "T", "NK", "B", "Myeloid", "Stromal", "Endothelial", "unclassified",
    # intermediate level
    "T_naive", "T_memory", "CD4_T", "CD8_T", "B_memory",
    # subtype: CD4 T cells
    "TN_CD4", "TCM_CD4", "TEM_CD4", "TEMRA_CD4", "Activated_CD4", "Treg", "TfH_like",
    # subtype: CD8 T cells
    "TN_CD8", "TCM_CD8", "TEM_CD8", "TEMRA_CD8", "Activated_CD8",
    # subtype: other T
    "T_terminal",
    # subtype: B cells (GC and Plasmablast are follicle-refined in place)
    "B_naive", "B_GC", "B_Plasmablast",
    # subtype: myeloid
    "Monocyte_Macrophage", "cDC1", "cDC2",
    # subtype: stromal / structural (only appear when not immune_only)
    "FDC", "Fibroblast", "Basement_Membrane",
    "Blood_Endothelial", "Lymphatic_Endothelial",
]


# Parent population for each row in the parent-relative heatmaps.
# Each cell type is divided by its parent's own total. The gating in
# celltype_rules_IHOPE.py is overlapping below the type level, so these
# rows are NOT expected to sum to 100 within a parent.
parent_of = {
    # percentage of total T
    "CD4_T": "T", "CD8_T": "T", "T_naive": "T", "T_memory": "T", "T_terminal": "T",
    # percentage of CD4 T
    "TN_CD4": "CD4_T", "TCM_CD4": "CD4_T", "TEM_CD4": "CD4_T", "TEMRA_CD4": "CD4_T",
    "Activated_CD4": "CD4_T", "Treg": "CD4_T", "TfH_like": "CD4_T",
    # percentage of CD8 T
    "TN_CD8": "CD8_T", "TCM_CD8": "CD8_T", "TEM_CD8": "CD8_T",
    "TEMRA_CD8": "CD8_T", "Activated_CD8": "CD8_T",
    # percentage of total B. GC and Plasmablast are gated off (B and not
    # naive), and Plasmablast is CD21- so it sits outside memory B, so
    # total B is the parent that actually contains all of them.
    "B_naive": "B", "B_memory": "B", "B_GC": "B", "B_Plasmablast": "B",
    # percentage of Myeloid.
    "Monocyte_Macrophage": "Myeloid", "cDC1": "Myeloid", "cDC2": "Myeloid",
}

# Root level. No single "Immune" row exists, so the denominator is the sum
# of these type-level rows, which is exactly total immune because the type
# level is a strict partition. Stromal, Endothelial and unclassified are
# excluded, so this is immune only with no unannotated cells.
immune_types = ["T", "B", "NK", "Myeloid"]


# Display names for the parent-relative heatmap row labels. Each row is
# labelled "child / parent" (e.g. "CD4 T cells / T cells"), so these are
# the presentation names, kept here in the notebook rather than the script.
celltype_display = {
    "T": "T cells", "B": "B cells", "NK": "NK cells", "Myeloid": "Myeloid cells",
    "CD4_T": "CD4 T cells", "CD8_T": "CD8 T cells",
    "T_naive": "Naive T cells", "T_memory": "Memory T cells",
    "T_terminal": "Terminally differentiated T cells",
    "TN_CD4": "Naive CD4 T cells", "TCM_CD4": "TCM CD4 T cells",
    "TEM_CD4": "TEM CD4 T cells", "TEMRA_CD4": "TEMRA CD4 T cells",
    "Activated_CD4": "Activated CD4 T cells", "Treg": "Regulatory T cells",
    "TfH_like": "TfH-like cells",
    "TN_CD8": "Naive CD8 T cells", "TCM_CD8": "TCM CD8 T cells",
    "TEM_CD8": "TEM CD8 T cells", "TEMRA_CD8": "TEMRA CD8 T cells",
    "Activated_CD8": "Activated CD8 T cells",
    "B_naive": "Naive B cells", "B_memory": "Memory B cells",
    "B_GC": "GC B cells", "B_Plasmablast": "Plasmablasts",
    "Monocyte_Macrophage": "Monocytes and macrophages",
    "cDC1": "cDC1", "cDC2": "cDC2",
}


Tissue order

In [ ]:
tissue_order = ["MedLN", "MesLN", "Spleen"]

In [ ]:
# Color palette for cell types

celltype_colors = {
    # type level
    "T": "#d62728",
    "B": "#1f77b4",
    "NK": "#2ca02c",
    "Myeloid": "#e377c2",
    "Stromal": "#8c564b",
    "Endothelial": "#9467bd",
    "unclassified": "#d9d9d9",

    # intermediate level
    "CD4_T": "#d62728",
    "CD8_T": "#ff7f0e",
    "T_naive": "#9467bd",
    "T_memory": "#2ca02c",
    "B_memory": "#1f77b4",

    # subtype level, B cells
    "B_naive": "#aec7e8",
    "B_GC": "#1f77b4",
    "B_Plasmablast": "#17becf",

    # subtype level, CD4 T cells
    "TN_CD4": "#d62728",
    "TCM_CD4": "#ff7f0e",
    "TEM_CD4": "#ff9896",
    "TEMRA_CD4": "#c5b0d5",
    "Activated_CD4": "#ad494a",
    "Treg": "#843c39",
    "TfH_like": "#e7298a",      # follicular, given a standout magenta

    # subtype level, CD8 T cells
    "TN_CD8": "#fdd0a2",        # changed from #ff7f0e to clear the collision
    "TCM_CD8": "#ffbb78",
    "TEM_CD8": "#bcbd22",
    "TEMRA_CD8": "#dbdb8d",
    "Activated_CD8": "#8c6d31",

    # subtype level, other T
    "T_terminal": "#7f7f7f",

    # subtype level, myeloid
    "cDC1": "#2ca02c",
    "cDC2": "#98df8a",
    "Monocyte_Macrophage": "#8c564b",

    # subtype level, stromal / structural
    "FDC": "#9467bd",
    "Fibroblast": "#c49c94",
    "Basement_Membrane": "#c7c7c7",
    "Blood_Endothelial": "#393b79",
    "Lymphatic_Endothelial": "#5254a3",
}

### Cell type matrices

Builds the full (all cell types), immune-only, and parent-relative matrices, for sample-level and tissue-level grouping. Pick which one to plot with `HEATMAP_NORM` in the Plots section below.

In [ ]:
matrix_all = pivot_for_heatmap(
    df, celltype_order=celltype_order, tissue_order=tissue_order,
    drop_unclassified=DROP_UNCLASSIFIED, renormalise=DROP_UNCLASSIFIED,
)
matrix_immune = pivot_for_heatmap(
    df, immune_only=True, renormalise=True,
    structural_cell_types=structural_cell_types,
    celltype_order=celltype_order, tissue_order=tissue_order,
)

matrix_tissue_all = pivot_for_tissue_heatmap(
    df_tissue, celltype_order=celltype_order,
    drop_unclassified=DROP_UNCLASSIFIED, renormalise=DROP_UNCLASSIFIED,
)
matrix_tissue_immune = pivot_for_tissue_heatmap(
    df_tissue, immune_only=True, renormalise=True,
    structural_cell_types=structural_cell_types,
    celltype_order=celltype_order,
)


# Parent-relative matrices. Each cell type divided by its parent population.
# Built from the plain all-types matrix, which already holds the parent rows.
matrix_parent = normalise_to_parent(
    pivot_for_heatmap(
        df, celltype_order=celltype_order, tissue_order=tissue_order,
        drop_unassigned=True,
    ),
    parent_of, immune_types,
)
matrix_tissue_parent = normalise_to_parent(
    pivot_for_tissue_heatmap(
        df_tissue, celltype_order=celltype_order, drop_unassigned=True,
    ),
    parent_of, immune_types,
)


## Plots

**Cell type heatmap**

Choose the normalisation with `HEATMAP_NORM` ("all", "immune", or "parent") and raw or log percentages with `USE_LOG` in the first block. Parent-relative mode expresses each cell type as a percentage of its parent population, fixes the colour scale at 0 to 100, ignores log scaling, and labels each row as "child / parent" (for example "CD4 T cells / T cells").

In [ ]:
USE_LOG = False           # log scaling, ignored when HEATMAP_NORM == "parent"
HEATMAP_NORM = "parent"   # "all", "immune", or "parent"
IMMUNE_ONLY = True        # used by the stacked barplots and stripplots below, not by the heatmaps

# Select the sample-level matrix for the heatmap and the clustermaps below.
if HEATMAP_NORM == "parent":
    matrix = matrix_parent
elif HEATMAP_NORM == "immune":
    matrix = matrix_immune
else:
    matrix = matrix_all

# Parent-relative values are already a percentage of a parent population,
# so log scaling does not apply and the colour scale is fixed at 0 to 100.
use_log = USE_LOG and HEATMAP_NORM != "parent"
scale_label = "log" if use_log else "linear"

if HEATMAP_NORM == "parent":
    cbar_label = "Percentage"
    heatmap_vmax = 100
else:
    cbar_label = "Log percentage (+0.1)" if use_log else "Percentage"
    heatmap_vmax = None

plot_matrix = np.log10(matrix + 0.1) if use_log else matrix

# Parent mode never contains an unclassified row, so this only acts on the
# all/immune matrices.
if HIDE_UNCLASSIFIED and HEATMAP_NORM != "parent":
    plot_matrix = plot_matrix.drop(index="unclassified", errors="ignore")

# A parent that is absent in a sample gives NaN (division by an empty
# parent). fillna(0) keeps the clustermap linkage happy; such cells render
# as 0. No-op for the all/immune matrices, which are already zero-filled.
plot_matrix = plot_matrix.fillna(0)

# In parent mode, relabel rows as "child / parent" so the denominator is
# visible per row. Applied last, on the plot copy only.
if HEATMAP_NORM == "parent":
    plot_matrix = parent_ratio_labels(
        plot_matrix, parent_of, immune_types, display_names=celltype_display,
    )


Generate plots:

In [ ]:
import importlib
from scripts import comparison
importlib.reload(comparison)
from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    print_numeric_summary,
    pivot_for_tissue_heatmap,
    plot_celltype_clustermap,
    plot_celltype_stacked_barplot,
    plot_celltype_stripplot,
    normalise_to_parent,
    parent_ratio_labels,
)


In [ ]:
plot_celltype_heatmap(
    plot_matrix,
    scale=scale_label,
    colorbar_legend=cbar_label,
    vmax=heatmap_vmax,
)


**Clustered with correlation metric**

This can be an advantage when sample sizes differ, which they do here

In [ ]:
plot_celltype_clustermap(
    plot_matrix,
    scale=scale_label,
    colorbar_legend=cbar_label,
    cluster_rows=False,
    cluster_cols=True,
    metric="correlation",
    vmax=heatmap_vmax,
)


**Tissue-grouped**

Select raw percentages or log-scaled in the first block.

In [ ]:
USE_LOG = False   # log scaling, ignored when HEATMAP_NORM == "parent"

# Reuses HEATMAP_NORM from the sample-level block above.
if HEATMAP_NORM == "parent":
    matrix_tissue = matrix_tissue_parent
elif HEATMAP_NORM == "immune":
    matrix_tissue = matrix_tissue_immune
else:
    matrix_tissue = matrix_tissue_all

use_log_tissue = USE_LOG and HEATMAP_NORM != "parent"
scale_label = "log" if use_log_tissue else "linear"

if HEATMAP_NORM == "parent":
    cbar_label = "Percentage"
    heatmap_vmax_tissue = 100
else:
    cbar_label = "Percentage (log +0.1)" if use_log_tissue else "Percentage"
    heatmap_vmax_tissue = None

plot_matrix_tissue = np.log10(matrix_tissue + 0.1) if use_log_tissue else matrix_tissue

if HIDE_UNCLASSIFIED and HEATMAP_NORM != "parent":
    plot_matrix_tissue = plot_matrix_tissue.drop(index="unclassified", errors="ignore")

plot_matrix_tissue = plot_matrix_tissue.fillna(0)

if HEATMAP_NORM == "parent":
    plot_matrix_tissue = parent_ratio_labels(
        plot_matrix_tissue, parent_of, immune_types, display_names=celltype_display,
    )


Raw percentages:

In [ ]:
plot_celltype_heatmap(
    plot_matrix_tissue,
    colorbar_legend=cbar_label,
    scale=scale_label,
    vmax=heatmap_vmax_tissue,
    narrow=True,
)


**Lymph nodes only**

In [ ]:
# Lymph-node-only heatmaps (MedLN + MesLN, spleen excluded)
# Reuses HEATMAP_NORM, USE_LOG, HIDE_UNCLASSIFIED, DROP_UNCLASSIFIED,
# celltype_order, structural_cell_types, parent_of and immune_types.

df_ln = df[df["tissue"] != "Spleen"]
df_tissue_ln = df_tissue[df_tissue["tissue"] != "Spleen"]

matrix_all_ln = pivot_for_heatmap(
    df_ln, celltype_order=celltype_order,
    drop_unclassified=DROP_UNCLASSIFIED, renormalise=DROP_UNCLASSIFIED,
)
matrix_immune_ln = pivot_for_heatmap(
    df_ln, immune_only=True, renormalise=True,
    structural_cell_types=structural_cell_types,
    celltype_order=celltype_order,
)
matrix_parent_ln = normalise_to_parent(
    pivot_for_heatmap(
        df_ln, celltype_order=celltype_order, drop_unassigned=True,
    ),
    parent_of, immune_types,
)

matrix_tissue_all_ln = pivot_for_tissue_heatmap(
    df_tissue_ln, celltype_order=celltype_order,
    drop_unclassified=DROP_UNCLASSIFIED, renormalise=DROP_UNCLASSIFIED,
)
matrix_tissue_immune_ln = pivot_for_tissue_heatmap(
    df_tissue_ln, immune_only=True, renormalise=True,
    structural_cell_types=structural_cell_types,
    celltype_order=celltype_order,
)
matrix_tissue_parent_ln = normalise_to_parent(
    pivot_for_tissue_heatmap(
        df_tissue_ln, celltype_order=celltype_order, drop_unassigned=True,
    ),
    parent_of, immune_types,
)

# Select the LN matrices to match HEATMAP_NORM.
if HEATMAP_NORM == "parent":
    matrix_ln = matrix_parent_ln
    matrix_tissue_ln = matrix_tissue_parent_ln
elif HEATMAP_NORM == "immune":
    matrix_ln = matrix_immune_ln
    matrix_tissue_ln = matrix_tissue_immune_ln
else:
    matrix_ln = matrix_all_ln
    matrix_tissue_ln = matrix_tissue_all_ln

use_log_ln = USE_LOG and HEATMAP_NORM != "parent"
scale_label_ln = "log" if use_log_ln else "linear"
if HEATMAP_NORM == "parent":
    cbar_label_ln = "Percentage"
    heatmap_vmax_ln = 100
else:
    cbar_label_ln = "Log percentage (+0.1)" if use_log_ln else "Percentage"
    heatmap_vmax_ln = None

plot_matrix_ln = np.log10(matrix_ln + 0.1) if use_log_ln else matrix_ln
if HIDE_UNCLASSIFIED and HEATMAP_NORM != "parent":
    plot_matrix_ln = plot_matrix_ln.drop(index="unclassified", errors="ignore")
plot_matrix_ln = plot_matrix_ln.fillna(0)
if HEATMAP_NORM == "parent":
    plot_matrix_ln = parent_ratio_labels(
        plot_matrix_ln, parent_of, immune_types, display_names=celltype_display,
    )

plot_matrix_tissue_ln = np.log10(matrix_tissue_ln + 0.1) if use_log_ln else matrix_tissue_ln
if HIDE_UNCLASSIFIED and HEATMAP_NORM != "parent":
    plot_matrix_tissue_ln = plot_matrix_tissue_ln.drop(index="unclassified", errors="ignore")
plot_matrix_tissue_ln = plot_matrix_tissue_ln.fillna(0)
if HEATMAP_NORM == "parent":
    plot_matrix_tissue_ln = parent_ratio_labels(
        plot_matrix_tissue_ln, parent_of, immune_types, display_names=celltype_display,
    )

plot_celltype_heatmap(
    plot_matrix_ln,
    scale=scale_label_ln,
    colorbar_legend=cbar_label_ln,
    vmax=heatmap_vmax_ln,
    title=" ",
)

# TODO maybe fix the scale bar for this one
plot_celltype_heatmap(
    plot_matrix_tissue_ln,
    scale=scale_label_ln,
    colorbar_legend=cbar_label_ln,
    vmax=heatmap_vmax_ln,
    title=" ",
    narrow=True,
)


### Stacked barplots

In [ ]:
LN_ONLY = True  # set True to restrict barplots to lymph node samples only (MedLN + MesLN, drop Spleen)

data_sample = df_ln if LN_ONLY else df
data_tissue = df_tissue_ln if LN_ONLY else df_tissue


Stacked barplot for broad/lineage cell types:

In [ ]:
palette_type = plot_celltype_stacked_barplot(
    data_sample,
    levels=["type"],
    immune_only=IMMUNE_ONLY,
    structural_cell_types=structural_cell_types,
    drop_unclassified=DROP_UNCLASSIFIED,
    tissue_order=tissue_order,
    ylim=(0,100)
)

plot_celltype_stacked_barplot(
    data_sample,
    levels=["type"],
    x="tissue",
    palette=palette_type,
    immune_only=IMMUNE_ONLY,
    structural_cell_types=structural_cell_types,
    drop_unclassified=DROP_UNCLASSIFIED,
    ylim=(0, 100)
)


**Subtype barplots:** Follows `IMMUNE_ONLY` and `LN_ONLY` above. When `IMMUNE_ONLY` is True, percentages are rescaled to 100% of immune cells only.

In [ ]:
palette_immune = plot_celltype_stacked_barplot(
    data_sample,
    levels=["subtype"],
    x="sample",
    immune_only=IMMUNE_ONLY,
    structural_cell_types=structural_cell_types,
    title="Immune cell composition per sample",
    tissue_order=tissue_order,
    ylim=(0, 100)
)

plot_celltype_stacked_barplot(
    data_tissue,
    levels=["subtype"],
    x="tissue",
    palette=palette_immune,
    structural_cell_types=structural_cell_types,
    immune_only=IMMUNE_ONLY,
    ylim=(0, 100)
)


Lineage breakdowns, each rescaled to sum to 100% within that lineage:

In [ ]:
plot_celltype_stacked_barplot(
    data_sample,
    levels=["subtype"],
    x="sample",
    lineage_subset="T",
    lineage_subsets=lineage_subsets,
    title="T cell composition per sample",
    ylabel="Percentage of T cells (CD45+, CD3e+)",
    ylim=(0, 100)
)

plot_celltype_stacked_barplot(
    data_sample,
    levels=["subtype"],
    x="sample",
    lineage_subset="B",
    lineage_subsets=lineage_subsets,
    title="B cell subtype composition per sample",
    ylabel="Percentage of B cells (CD45+, CD20+/CD79a+)",
    ylim=(0, 100)
)

plot_celltype_stacked_barplot(
    data_sample,
    levels=["subtype"],
    x="sample",
    lineage_subset="Myeloid",
    lineage_subsets=lineage_subsets,
    title="DC and monocyte/macrophage subtype composition per sample",
    ylabel="Percentage of myeloid cells (CD45+, CD11+/HLA-DR+)",
    ylim=(0, 100)
)

In [ ]:
plot_celltype_stacked_barplot(
    data_tissue,
    levels=["subtype"],
    x="tissue",
    lineage_subset="T",
    lineage_subsets=lineage_subsets,
    title="T cell composition per tissue",
    ylabel="Percentage of T cells (CD45+, CD3e+)",
    ylim=(0, 100)
)

plot_celltype_stacked_barplot(
    data_tissue,
    levels=["subtype"],
    x="tissue",
    lineage_subset="B",
    lineage_subsets=lineage_subsets,
    title="B cell subtype composition per tissue",
    ylabel="Percentage of B cells (CD45+, CD20+/CD79a+)",
    ylim=(0, 100)
)

plot_celltype_stacked_barplot(
    data_tissue,
    levels=["subtype"],
    x="tissue",
    lineage_subset="Myeloid",
    lineage_subsets=lineage_subsets,
    title="DC and monocyte/macrophage subtype composition per tissue",
    ylabel="Percentage of myeloid cells (CD45+, CD11+/HLA-DR+)",
    ylim=(0, 100)
)

**Updated** T cell subtype barplots: CD8+/CD4+ split

In [ ]:
lineage_subsets["CD4_T"] = ["TN_CD4", "TCM_CD4", "TEM_CD4", "TEMRA_CD4"]
lineage_subsets["CD8_T"] = ["TN_CD8", "TCM_CD8", "TEM_CD8", "TEMRA_CD8"]

In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="sample",
    lineage_subset="CD4_T",
    lineage_subsets=lineage_subsets,
    title="CD4 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD4", "TN_CD4", "TEM_CD4", "TCM_CD4"],
    palette=celltype_colors,
)

In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="tissue",
    lineage_subset="CD4_T",
    lineage_subsets=lineage_subsets,
    title="CD4 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD4", "TN_CD4", "TEM_CD4", "TCM_CD4"],
    palette=celltype_colors,
)

In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="sample",
    lineage_subset="CD8_T",
    lineage_subsets=lineage_subsets,
    title="CD8 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD8", "TN_CD8", "TEM_CD8", "TCM_CD8"],
    palette=celltype_colors,
)

In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="tissue",
    lineage_subset="CD8_T",
    lineage_subsets=lineage_subsets,
    title="CD8 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD8", "TN_CD8", "TEM_CD8", "TCM_CD8"],
    palette=celltype_colors,
)

**Donor-specified strip plots**

In [ ]:
cd4_types = ["Activated_CD4", "Treg"]
cd8_types = ["Activated_CD8"]

df_cd4 = df[df["cell_type"].isin(cd4_types)]
df_cd8 = df[df["cell_type"].isin(cd8_types)]

plot_celltype_stripplot(
    df_cd4,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
)

plot_celltype_stripplot(
    df_cd8,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
)

In [ ]:
plot_celltype_stripplot(
    df,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    cell_types=["Activated_CD4", "Treg"],
    denominator_cell_type="CD4_T",
)

plot_celltype_stripplot(
    df,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    cell_types=["Activated_CD8"],
    denominator_cell_type="CD8_T",
)

In [ ]:
plot_celltype_stripplot(
    df,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    #cell_types=["Activated_CD4", "Treg", "Activated_CD8"],
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
)

With Mann-Whitney U test

In [ ]:
plot_celltype_stripplot(
    df,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    celltype_order=celltype_order,
    drop_unclassified=DROP_UNCLASSIFIED,
    summary="bar",
    mannwhitney=True,
)

New versions with separate LN/spleens

In [ ]:
df_ln = df[df["tissue"] != "Spleen"]

plot_celltype_stripplot(
    df_ln,
    level="intermediate",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    celltype_order=celltype_order,
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
    summary="bar",
)

plot_celltype_stripplot(
    df_ln,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    celltype_order=celltype_order,
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
    summary="bar",
)

In [ ]:
import importlib
from scripts import comparison
importlib.reload(comparison)
from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    print_numeric_summary,
    pivot_for_tissue_heatmap,
    plot_celltype_clustermap,
    plot_celltype_stacked_barplot,
    plot_celltype_stripplot,
    normalise_to_parent,
    parent_ratio_labels,
)


In [ ]:
df_spleen = df[df["tissue"] == "Spleen"]

plot_celltype_stripplot(
    df_spleen,
    level="intermediate",
    x="tissue",
    tissue_order=["Spleen"],
    donor_colors=donor_colors,
    celltype_order=celltype_order,
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
    summary="bar",
    reference_df=df_ln,
    reference_group_col="tissue",
)

plot_celltype_stripplot(
    df_spleen,
    level="subtype",
    x="tissue",
    tissue_order=["Spleen"],
    donor_colors=donor_colors,
    celltype_order=celltype_order,
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
    summary="bar",
    reference_df=df_ln,
    reference_group_col="tissue",
)

## Differential screen

Compares fold change (FC) and percentage point difference between mes/med LN.

In [ ]:
import importlib
from scripts import differential_screen as ds
importlib.reload(ds)

reports_dir = "/Users/emmyberg/Documents/IHOPE_SpatialProteomics/results/reports/zscore_log2"
anndata_dir = "/Users/emmyberg/Documents/IHOPE_SpatialProteomics/data/anndata/zscore_log2/celltyped/follicledomains"

basenames = [
    "IHOPE14_MedLN_BottomLeft", "IHOPE14_MedLN_TopRight", "IHOPE14_MedLN_BottomRight",
    "IHOPE14_MesLN", "IHOPE20_MedLN", "IHOPE20_Spleen", "IHOPE26_MedLN",
    "IHOPE26_Spleen", "IHOPE27_MedLN", "IHOPE27_Spleen", "IHOPE39_MedLN",
    "IHOPE39_MesLN_A", "IHOPE39_MesLN_B", "IHOPE39_Spleen",
]

parent_of = {
    # percentage of total T
    "CD4_T": "T", "CD8_T": "T", "T_naive": "T", "T_memory": "T", "T_terminal": "T",
    # percentage of CD4 T
    "TN_CD4": "CD4_T", "TCM_CD4": "CD4_T", "TEM_CD4": "CD4_T", "TEMRA_CD4": "CD4_T",
    "Activated_CD4": "CD4_T", "Treg": "CD4_T", "TfH_like": "CD4_T",
    # percentage of CD8 T
    "TN_CD8": "CD8_T", "TCM_CD8": "CD8_T", "TEM_CD8": "CD8_T",
    "TEMRA_CD8": "CD8_T", "Activated_CD8": "CD8_T",
    # percentage of total B
    "B_naive": "B", "B_memory": "B", "B_GC": "B", "B_Plasmablast": "B",
    # percentage of Myeloid
    "Monocyte_Macrophage": "Myeloid", "cDC1": "Myeloid", "cDC2": "Myeloid",
}

Cell type abundance

In [ ]:
long_df = ds.load_celltype_long(reports_dir, basenames)
donor = ds.pool_to_donor_level(long_df)             # pct over all cells (raw, kept)
donor = ds.exclude_unresolved(donor)                # drops _unclassified and _unassigned, adds pct_resolved
donor = ds.add_parent_relative(donor, parent_of)    # pct_parent, invariant to the base

abund = ds.screen(donor, group_a="MesLN", group_b="MedLN", value_col="pct_resolved")
abund_parent = ds.screen(donor, group_a="MesLN", group_b="MedLN", value_col="pct_parent")

Marker positivity

In [ ]:
positivity_csv = "marker_positivity_by_sample.csv"
ds.extract_marker_positivity(anndata_dir, basenames, positivity_csv)  # comment out after first run
marker_donor = ds.marker_to_donor_level(positivity_csv)
markers = ds.screen(marker_donor, group_a="MesLN", group_b="MedLN", value_col="pct")

FC plots with average values

In [ ]:
ds.plot_fc_lollipop(abund, metric="log2fc", points="group", top_n=12,
                    title="MesLN vs MedLN abundance, fold change")
ds.plot_fc_lollipop(abund, metric="diff", points="group", top_n=12,
                    title="MesLN vs MedLN abundance, percentage points")
ds.plot_fc_lollipop(abund_parent, metric="log2fc", points="group", top_n=12,
                    title="MesLN vs MedLN, parent-relative fold change")
ds.plot_fc_lollipop(markers, metric="log2fc", points="group", top_n=15,
                    title="MesLN vs MedLN marker positivity, fold change")

Per-donor FC plots

In [ ]:
import importlib
from scripts import differential_screen as ds
importlib.reload(ds)

abund = ds.screen(donor, group_a="MesLN", group_b="MedLN", value_col="pct_resolved")
abund_parent = ds.screen(donor, group_a="MesLN", group_b="MedLN", value_col="pct_parent")
markers = ds.screen(marker_donor, group_a="MesLN", group_b="MedLN", value_col="pct")

# with dot size scaled by abundance
ds.plot_fc_lollipop(abund, metric="log2fc", points="donor", size_by_abundance=True, top_n=12,
                    title="MesLN vs MedLN abundance, fold change per donor")
ds.plot_fc_lollipop(abund_parent, metric="log2fc", points="donor", size_by_abundance=True, top_n=12,
                    title="MesLN vs MedLN, parent-relative fold change per donor")
ds.plot_fc_lollipop(markers, metric="log2fc", points="donor", size_by_abundance=True, top_n=15,
                    title="MesLN vs MedLN marker positivity per donor")

# same, without abundance sizing (uniform dots)
ds.plot_fc_lollipop(abund, metric="log2fc", points="donor", size_by_abundance=False, top_n=12,
                    title="MesLN vs MedLN abundance, fold change per donor")
                    title="MesLN vs MedLN, parent-relative fold change per donor")
ds.plot_fc_lollipop(markers, metric="log2fc", points="donor", size_by_abundance=False, top_n=15,
                    title="MesLN vs MedLN marker positivity per donor")

**Separate FC plots for CD4/CD8 and manual marker lists**

In [ ]:
# CD4 and CD8 T subsets, matching lineage_subsets in the other notebook
cd4_subsets_naive = ["TN_CD4", "TCM_CD4", "TEM_CD4", "TEMRA_CD4"]
cd8_subsets_naive = ["TN_CD8", "TCM_CD8", "TEM_CD8", "TEMRA_CD8"]
cd4_subsets = ["TCM_CD4", "TEM_CD4", "TEMRA_CD4"]   # naive dropped
cd8_subsets = ["TCM_CD8", "TEM_CD8", "TEMRA_CD8"]

# Union of the DE-top list and the panel list (DE list plus markers only in the panel list)
expanded_markers = [
    # DE-top markers
    "CD11c", "CD1c", "Collagen IV", "HLA-DR", "TCF-1", "CXCL13", "CCR7",
    "CD20", "CD68", "CD107a", "CD141", "CD38", "FOXP3", "IFNG", "Ki67",
    # added from the panel list
    "CCR6", "CD27", "CD45RA", "CD57", "CD69", "ICOS", "Granzyme B", "PD-1", "CD45RO",
]

In [ ]:
plt.style.use("default")

# with naive included
ds.plot_fc_lollipop(
    abund_parent[abund_parent["cell_type"].isin(cd4_subsets_naive)],
    metric="log2fc", points="donor", size_by_abundance=False, top_n=len(cd4_subsets_naive),
    title="MesLN vs MedLN, CD4 T subsets (parent-relative), per donor",
)
ds.plot_fc_lollipop(
    abund_parent[abund_parent["cell_type"].isin(cd8_subsets_naive)],
    metric="log2fc", points="donor", size_by_abundance=False, top_n=len(cd8_subsets_naive),
    title="MesLN vs MedLN, CD8 T subsets (parent-relative), per donor",
)

# without naive
ds.plot_fc_lollipop(
    abund_parent[abund_parent["cell_type"].isin(cd4_subsets)],
    metric="log2fc", points="donor", size_by_abundance=False, top_n=len(cd4_subsets),
    title="MesLN vs MedLN, CD4 memory T subsets (parent-relative), per donor",
)
ds.plot_fc_lollipop(
    abund_parent[abund_parent["cell_type"].isin(cd8_subsets)],
    metric="log2fc", points="donor", size_by_abundance=False, top_n=len(cd8_subsets),
    title="MesLN vs MedLN, CD8 memory T subsets (parent-relative), per donor",
)

In [ ]:
ds.plot_fc_lollipop(
    markers[markers["cell_type"].isin(expanded_markers)],
    metric="log2fc", points="donor", size_by_abundance=False, top_n=len(expanded_markers),
    title="MesLN vs MedLN marker positivity per donor",
)

Summary tables

In [ ]:
pd.set_option("display.max_rows", None, "display.width", None)

print("ABUNDANCE (percentage of resolved cells, per level)")
print(abund.to_string(index=False))

print("\nABUNDANCE parent-relative")
print(abund_parent.to_string(index=False))

print("\nMARKER positivity")
print(markers.to_string(index=False))